# Prototype of Improving LLM-Based Service Composition with symbolic reasoning


In [ ]:
# imports
import os
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path
import json

# Loads .env variables into environment.
load_dotenv()

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY")
)

Loads the benchmarks. You may specify what benchmark types you'd like to use in BENCHMARK_TYPES.

In [30]:
BENCHMARK_DIR = Path("./benchmark")
BENCHMARK_TYPES = ["socbenchd_1"]  # add more benchmark types here
BENCHMARK_LIMIT = 1   # number of sectors to load per type, set to None for all
QUERY_LIMIT = 1       # number of queries to load per sector, set to None for all

benchmark_sets = []

for benchmark_type in BENCHMARK_TYPES:
    sectors = sorted(p for p in (BENCHMARK_DIR / benchmark_type).iterdir() if p.is_dir())
    for sector_path in sectors[:BENCHMARK_LIMIT]:
        queries = json.loads((sector_path / "queries.json").read_text())["queries"]
        benchmark_sets.append({
            "name": sector_path.name,
            "type": benchmark_type,
            "services": [p.read_text() for p in sector_path.rglob("openapi.json")],
            "queries": queries[:QUERY_LIMIT]
        })

total_available = sum(
    len([p for p in (BENCHMARK_DIR / t).iterdir() if p.is_dir()])
    for t in BENCHMARK_TYPES
)
total_queries = sum(len(b["queries"]) for b in benchmark_sets)

print(f"Loaded {len(benchmark_sets)} benchmark sets (total available: {total_available})")
print(f"Total queries: {total_queries}")

Loaded 1 benchmark sets (total available: 11)
Total queries: 1


In [23]:
# calls the LLM
def call_llm(prompt: str, model: str, instructions: str) -> str:
    response = client.responses.create(
        model=model,
        instructions=instructions,
        input=prompt
    )

    return response.output_text

In [ ]:
def build_prompt(services: list[str], query: str) -> str:
    services_block = "\n\n---\n\n".join(services)

    return f"""You are given a set of REST API specifications and a task description.
Your job is to write Python code using the `requests` library that fulfills the task by calling the necessary endpoints in the correct order.

Rules:
- Only use endpoints defined in the provided specifications.
- Pass outputs from earlier calls as inputs to later ones where needed.
- Use placeholder values (e.g. "YOUR_API_KEY") for any required authentication.
- Return ONLY raw Python code. No markdown, no code fences, no comments, no notes, no explanations — nothing but the code itself.
- All the code shall be under a function called compose.

## API Specifications

{services_block}

## Task

{query}

## Python Code
"""

In [31]:
MODEL = "NousResearch/Hermes-4-70B"

for benchmark in benchmark_sets:
    for query in benchmark["queries"]:
        prompt = build_prompt(benchmark["services"], query["query"])
        result = call_llm(prompt, MODEL, "Return only Python code, no explanation.")
        print(f"=== {benchmark['type']} / {benchmark['name']} ===")
        print(f"Query: {query['query']}\n")
        print(result)
        print()

=== socbenchd_1 / 01-energy ===
Query: Retrieve the status and performance metrics of all monitored energy equipment, access active alerts for any system performance issues, analyze the impact of weather conditions on electricity demand between specific dates, configure new alert thresholds for unusual energy generation patterns in specific sectors, and submit the integration status of renewable energy sources such as solar or wind into the energy grid.

import requests

def compose():
    equipment_status_response = requests.get('https://api.energysector.com/equipment-monitoring')
    
    active_alerts_response = requests.get('https://api.energysector.com/alerts')
    
    weather_impact_analysis_response = requests.get('https://api.energysector.com/weather-impact-analysis?start_date=2023-01-01&end_date=2023-12-31')
    
    configure_alerts_payload = {
        "alert_types": ["overload", "low_supply", "demand_spike"],
        "email": "alerts@example.com",
        "thresholds": {
  